%md
#### Goal
- To create a dataobject to capture the party alternative names

#### Authors
- Abhishek.R.Jaiswal@rabobank.com

##### Flow of Logic
- Reading GCOB,GCDS,GIC dataobjects from GDP
- Fetching the alternative names from different SourceSystems
- Loading the final dataobject to SA RADAR Storage account

##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek Jaiswal	 |17-Sept-205 |12646357 |First Release
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
| Mahalakshmi V | 20-May-2026 | 16070024 | Included RANZ V4


#####READING FILES FROM GDP

In [0]:
import os
import pandas as pd
from datetime import datetime, timedelta

In [0]:
from RadarUtils import *

In [0]:
app_reg_app_id = os.environ["APP_REG_APP_ID"]
ReadStorage = os.environ["GDP_STORAGE_NAME"]
TenantId = os.environ["TENANT_ID"]
GIC_ReadStorage = os.environ["GDP_SA_STORAGE_NAME"]
RANZ_ReadStorage = os.environ['AU_GDP_Defined_Storage_Account']
environment = os.environ["ENV"]
radar_datamodel_version_number = 3

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_AlternativeName_dataobject = "Party_AlternativeNames"

In [0]:
SARADAR = "saradar" + environment

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(SARADAR)
authenticate_storage_account(RANZ_ReadStorage)

In [0]:
# Derive the date for which data has to be processed from GDP
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# List of datasets from GDP
gcds_df = pd.DataFrame(
    {
        "definedDatasetname": [
            "client_Client",
            "client_KeyStoreKey"
        ]
    }
)

# Create TempView for each loading table
for index, row in gcds_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=row.definedDatasetname,Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
            "party_case_client_details",
            "party_AllPartyDetails",
            "party_Alias",
            "party_trade_name",
            "party_client_structure_GUI"
        ]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier = spark.read.parquet(
    f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet"
)

In [0]:
# List of datasets from GDP
load_df = [
'Legacy2_case_client_details'
,'Legacy2_trade_name'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
load_df = [
    'c_b_party_xref',
    'c_b_contr_rol_party_xref',
    'c_b_organisation_xref',
    'c_b_individual_xref',
]
# Run the load
load_ranz_v4_rdm_tables(load_df, Load_Date)

#####TRANSFORMATION TO GET Party_AlternativeNames OBJECT 

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select distinct *
,case when ClientType = 'Legal Entity' then 'Legal Entity'
  when ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person'
  else null
end as Party_type
,Case when ClientType = 'Legal Entity' then concat('GCOB_LEC_', GcobId)
  when ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)') then concat('GCOB_NP_NPPC_', GcobId)
End as LocalSystemIdentifier
from party_case_client_details
where CaseStatusName <> 'Cancelled'

In [0]:
%sql
CREATE OR REPLaCE TEMP VIEW Gcob_AllPartyDetails as
with PartyDetails as 
(
Select distinct p.*
,Case when p.ClientType = 'LegalEntityClient' then concat('LEC_',Id)
      when p.ClientType in ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)') then concat('NP_NPPC_',Id)
End as SourceClient 
,case when p.CaseStatusName is null then 'RelatedParty' else c.CaseStatusName end as StatusName
,case when p.ClientType = 'LegalEntityClient' then 'Legal Entity' 
   when p.ClientType in ('NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person'
   when p.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity'
   when p.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person'
   else null
end as Party_type
from party_AllPartyDetails p
LEFT JOIN party_case_client_details c on p.UniquePartyId = c.UniqueGcobId
where Status = 'Snapshot' 
)
,AllPartyDetails as
(
select * from PartyDetails where StatusName not in ('Cancelled','RelatedParty')
)
,Client_MaxSnap as
(
select papd.UniquePartyId,max(CS.ClientStructureSnapshotId) as Max_SnapshotId
FROM PartyDetails papd 
LEFT JOIN party_client_structure_GUI cs ON papd.Sourceclient=cs.Sourceclient
where papd.StatusName <> 'Cancelled'
group by papd.UniquePartyId
)
,RP_MaxSnap as
(
select papd.UniquePartyId,max(CS.ClientStructureSnapshotId) as Max_SnapshotId
FROM PartyDetails papd 
LEFT JOIN party_client_structure_GUI cs ON papd.PartyId = cs.ParentEntityId and papd.UniquePartyId = cs.UniqueParentPartyId
where papd.StatusName <> 'Cancelled'
group by papd.UniquePartyId
)
,Client as 
(
select distinct
papd.GcobId
,papd.UniquePartyId
,CS.ClientStructureSnapshotId
,NULLIF(alias.Alias, '  ') as Alias
,tr.ClientTradeName as TradeName
,NULLIF(papd.FullLegalNameInLocalLanguage, '') as FullLegalNameInLocalLanguage
,papd.Party_type
,papd.PartyId
,papd.FirstName , papd.MiddleName , papd.LastName
FROM AllPartyDetails papd 
LEFT JOIN party_client_structure_GUI cs ON papd.Sourceclient=cs.Sourceclient
LEFT JOIN party_Alias alias ON papd.Id = alias.Id AND papd.ClientType = alias.ResultType --papd.PartyId = alias.PartyId AND
LEFT JOIN party_trade_name tr ON papd.Id = tr.Id AND papd.ClientType = tr.ResultType --papd.PartyId = tr.PartyId AND
)
,Client_Details as
(
select distinct c.GcobId,c.UniquePartyId,c.Alias,c.TradeName,c.FullLegalNameInLocalLanguage,c.Party_type,c.PartyId
,c.FirstName , c.MiddleName , c.LastName
from Client c
inner join Client_MaxSnap m on c.UniquePartyId = m.UniquePartyId and COALESCE(c.ClientStructureSnapshotId,00) = COALESCE(m.Max_SnapshotId,00)
where Alias is not null or TradeName is not null or FullLegalNameInLocalLanguage is not null or c.FirstName is not null or c.MiddleName is not null or c.LastName is not null
)
,RelatedPrty as 
(
select distinct
papd.GcobId
,papd.UniquePartyId
,CS.ClientStructureSnapshotId
,NULLIF(alias.Alias, '  ') as Alias
,tr.ClientTradeName as TradeName
,NULLIF(papd.FullLegalNameInLocalLanguage, '') as FullLegalNameInLocalLanguage
,case when papd.ClientType = 'LegalEntityClient' then 'Legal Entity' 
   when papd.ClientType in ('NaturalPersonClient', 'Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person'
   when papd.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity'
   when papd.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person'
   else null
end as Party_type
,papd.PartyId
,papd.FirstName , papd.MiddleName , papd.LastName
FROM party_client_structure_GUI cs
LEFT JOIN party_AllPartyDetails papd ON cs.ParentEntityId = papd.PartyId and cs.UniqueParentPartyId=papd.UniquePartyId
LEFT JOIN party_Alias alias ON papd.Id = alias.Id AND papd.ClientType = alias.ResultType --papd.PartyId = alias.PartyId AND
LEFT JOIN party_trade_name tr ON papd.Id = tr.Id AND papd.ClientType = tr.ResultType --papd.PartyId = tr.PartyId AND
where papd.Status='Snapshot' 
)
,RelatedPrty_Details as
(
select distinct c.GcobId,c.UniquePartyId,c.Alias,c.TradeName,c.FullLegalNameInLocalLanguage,c.Party_type,c.PartyId
,c.FirstName , c.MiddleName , c.LastName
from RelatedPrty c
inner join RP_MaxSnap m on c.UniquePartyId = m.UniquePartyId and COALESCE(c.ClientStructureSnapshotId,00) = COALESCE(m.Max_SnapshotId,00)
where Alias is not null or TradeName is not null or FullLegalNameInLocalLanguage is not null or c.FirstName is not null or c.MiddleName is not null or c.LastName is not null
)
,Client_RelatedPrty as 
(
select * from Client_Details
union
select * from RelatedPrty_Details
)
select distinct 
CONCAT('GCOB_', UniquePartyId) AS LocalSystemIdentifier
,GcobId,UniquePartyId,Alias,TradeName,FullLegalNameInLocalLanguage,Party_type,PartyId
,FirstName , MiddleName , LastName
from Client_RelatedPrty


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)
;

CREATE OR REPLACE TEMPORARY VIEW Legacy2 AS
with L2_Details as
(
select distinct 
c.clientId,SalesforceClientID_nCino
,CASE
  WHEN c.IsClient = 'true' and  c.ClientTypeId = 1 THEN concat('LE_', GcobId)
  WHEN c.IsClient = 'true' and c.ClientTypeId in (2,3) THEN concat('NP_NPPC_', GcobId)
  WHEN c.IsClient = 'false' and c.ClientTypeId = 1 THEN concat('RLE_', GcobId)
  WHEN c.IsClient = 'false' and c.ClientTypeId in (2,3) THEN concat('RNP_', GcobId)
END as Legacy2_Identifier
,case when c.ClientType = 'Legal Entity' then 'Legal Entity' 
      when c.ClientType in ('Natural Person','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      when c.ClientType = 'Related Legal Entity' then 'Related Legal Entity' 
      when c.ClientType = 'Related Natural Person' then 'Related Natural Person' 
      else null
end as Party_type
,concat('LEGACY2_',Legacy2_Identifier) as LocalSystemIdentifier
,c.FullLegalNameLocalLanguage as FullLegalNameInLocalLanguage 
,tr.TradeName
,case when c.ClientType not in ('Related Legal Entity','Legal Entity') then ContactFirstName else null end as FirstName
,case when c.ClientType not in ('Related Legal Entity','Legal Entity') then ContactMiddleName else null end as MiddleName
,case when c.ClientType not in ('Related Legal Entity','Legal Entity') then ContactLastName else null end as LastName
from Legacy2_client c
LEFT JOIN Legacy2_trade_name tr on c.ClientId = tr.ClientId
where ClientTypeId in (1,2,3)
)
select * from L2_Details
where FullLegalNameInLocalLanguage is not null or FirstName is not null or MiddleName is not null or LastName is not null or TradeName is not null

In [0]:
%sql
create or Replace Temporary view Ranz_AlternativeNames as
select distinct
CONCAT('RANZ_',cbp.SRC_PARTY_ID) LocalSystemIdentifier,
cbp.SRC_PARTY_ID,
'RANZ' as Application,
null FullLegalNameInLocalLanguage,
-- CONCAT_WS(',',PREV_NAME,PREF_NAME,OTH_KWN_FIRST_NAME,OTH_KWN_MIDDLE_NAME,OTH_KWN_LAST_NAME) AS Alias,
PREF_NAME as Alias,
null as TradeName,
ind.FIRST_NAME as FirstName,
ind.MIDDLE_NAME as MiddleName,
ind.LAST_NAME as LastName
FROM c_b_party as cbp
LEFT OUTER JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = cbp.ROWID_XREF
LEFT OUTER JOIN c_b_individual ind
   ON ind.FK_PARTY_ID= cbp.ROWID_XREF

union

select distinct
CONCAT('RANZ_',cbp.SRC_PARTY_ID) LocalSystemIdentifier,
cbp.SRC_PARTY_ID,
'RANZ' as Application,
null FullLegalNameInLocalLanguage,
null as Alias,
org.ORG_TRADE_NAME as TradeName,
null as FirstName,
null as MiddleName,
null as LastName
FROM c_b_party as cbp
LEFT OUTER JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = cbp.ROWID_XREF
LEFT OUTER JOIN c_b_organisation org
    ON org.FK_PARTY_ID= cbp.ROWID_XREF

In [0]:
%sql
Create or replace temporary view GCDS_Clients As
with gcds as
(
select distinct
k.KeyStore_value as identifier
,k.KeyStore_type
,c.Party_Type
,c.GCID
,c.Trade_name as TradeName
,case when c.Party_type <> 'Natural Person' then c.Local_name else c.Person_LocalName end FullLegalNameInLocalLanguage
from client_KeyStoreKey k
inner join client_Client c on c.GCID = k.GCID
)
select * from gcds where TradeName is not null or FullLegalNameInLocalLanguage is not null

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Details_GCDS AS
SELECT distinct
CASE WHEN gcds.identifier = gcob.GcobId AND gcob.Party_type = 'Legal Entity' AND gcds.KeyStore_type = 'GCOBID' 
     THEN gcob.LocalSystemIdentifier
    WHEN c.ClientId = ncino.PartyId AND ncino.Party_type = 'Natural Person' THEN ncino.LocalSystemIdentifier
    WHEN l2.SalesforceClientID_nCino = gcds.identifier AND l2.Party_type = gcds.Party_type AND gcds.KeyStore_type = 'NCINOID' THEN l2.LocalSystemIdentifier
    WHEN gcds.identifier = Ranz_AlternativeNames.SRC_PARTY_ID and gcds.KeyStore_type = 'CB RANZ' THEN Ranz_AlternativeNames.LocalSystemIdentifier
    else CONCAT('GCDS_', gcds.GCID)
END AS LocalSystemIdentifier
,CASE WHEN gcds.identifier = gcob.GcobId AND gcob.Party_type = 'Legal Entity' AND gcds.KeyStore_type = 'GCOBID' THEN 'GCOB'
    WHEN c.ClientId = ncino.PartyId AND ncino.Party_type = 'Natural Person' THEN 'GCOB'
    WHEN l2.SalesforceClientID_nCino = gcds.identifier AND l2.Party_type = gcds.Party_type AND gcds.KeyStore_type = 'NCINOID'
    THEN 'Legacy2'
    WHEN gcds.identifier = Ranz_AlternativeNames.SRC_PARTY_ID and gcds.KeyStore_type = 'CB RANZ' THEN 'RANZ'
    Else 'GCDS'
END AS Application
,COALESCE(gcds.FullLegalNameInLocalLanguage,gcob.FullLegalNameInLocalLanguage, ncino.FullLegalNameInLocalLanguage, l2.FullLegalNameInLocalLanguage) AS FullLegalNameInLocalLanguage
,COALESCE(gcob.Alias, ncino.Alias,Ranz_AlternativeNames.Alias) AS Alias
,COALESCE(gcds.TradeName,gcob.TradeName, ncino.TradeName, l2.TradeName,Ranz_AlternativeNames.TradeName) AS TradeName
,COALESCE(gcob.FirstName, ncino.FirstName, l2.FirstName,Ranz_AlternativeNames.FirstName) AS FirstName
,COALESCE(gcob.MiddleName, ncino.MiddleName, l2.MiddleName,Ranz_AlternativeNames.MiddleName) AS MiddleName
,COALESCE(gcob.LastName, ncino.LastName, l2.LastName,Ranz_AlternativeNames.LastName) AS LastName
FROM GCDS_cLIENTS gcds
LEFT JOIN Gcob_AllPartyDetails gcob ON gcds.identifier = gcob.GcobId AND gcob.Party_type = 'Legal Entity' 
  AND gcds.KeyStore_type = 'GCOBID'
LEFT JOIN Gcob_CaseClientDetails c ON c.SalesforceClientID_nCino = gcds.identifier AND gcds.KeyStore_type = 'NCINOID'
LEFT JOIN Gcob_AllPartyDetails ncino ON c.ClientId = ncino.PartyId AND ncino.Party_type = 'Natural Person'
LEFT JOIN Legacy2 l2 on l2.SalesforceClientID_nCino = gcds.identifier and l2.Party_type = gcds.Party_type 
  and gcds.KeyStore_type = 'NCINOID'
LEFT JOIN Ranz_AlternativeNames  on gcds.identifier = Ranz_AlternativeNames.SRC_PARTY_ID 
         and gcds.KeyStore_type = 'CB RANZ'   

In [0]:
%sql
Create or replace temporary view Details_NonGCDS As
with NonGCDS as (
select distinct
LocalSystemIdentifier
,'GCOB' as Application
,FullLegalNameInLocalLanguage
,Alias
,TradeName
,FirstName 
,MiddleName 
,LastName
from Gcob_AllPartyDetails

union

select distinct
t1.LocalSystemIdentifier
,'Legacy2' as Application
,t1.FullLegalNameInLocalLanguage
,t1.TradeName as Alias
,null as TradeName
,t1.FirstName 
,t1.MiddleName 
,t1.LastName
From Legacy2 t1

union
select distinct
LocalSystemIdentifier
,Application
,FullLegalNameInLocalLanguage
,Alias
,TradeName
,FirstName 
,MiddleName 
,LastName
from
Ranz_AlternativeNames
)
select distinct
LocalSystemIdentifier
,Application
,FullLegalNameInLocalLanguage
,Alias
,TradeName
,FirstName 
,MiddleName 
,LastName
from NonGCDS 

In [0]:
%sql
Create or replace temporary view NonGCDS_UniqueDetails As
select * 
from Details_NonGCDS a
left anti join Details_GCDS b
on a.LocalSystemIdentifier = b.LocalSystemIdentifier


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW AlternativeNames AS
SELECT * FROM Details_GCDS WHERE Application IS NOT NULL
UNION
select * from NonGCDS_UniqueDetails

In [0]:
df_AlternativeNames = spark.table("AlternativeNames")

In [0]:
df_f_AlternativeNames = add_party_identifier(df_AlternativeNames, df_Party_SystemIdentifier)

In [0]:
df_f_AlternativeNames.createOrReplaceTempView('Alternative')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Party_AlternativeNames AS
select distinct
PartyIdentifier
,Application as SourceApplication
,'FullLegalNameInLocalLanguage' as PartyNameTypeCode
,FullLegalNameInLocalLanguage as FullNameValue
,FirstName 
,MiddleName 
,LastName
from Alternative
where FullLegalNameInLocalLanguage is not null or FirstName is not null or MiddleName is not null or LastName is not null

UNION

select distinct
PartyIdentifier
,Application as SourceApplication
,'Alias' as PartyNameTypeCode
,Alias as FullNameValue
,null as FirstName 
,null as MiddleName 
,null as LastName
from Alternative
where Alias is not null

UNION

select distinct
PartyIdentifier
,Application as SourceApplication
,'TradeName' as PartyNameTypeCode
,TradeName as FullNameValue
,null as FirstName 
,null as MiddleName 
,null as LastName
from Alternative
where TradeName is not null

##### CREATING A FINAL DATAFRAME BY COMBINING ALL THE INFORMATION FROM DIFFERENT SOURCESYSTEMS AND LOADING THE FINAL DATAOBJECT TO SA RADAR STORAGE ACCOUNT

In [0]:
df_final_AlternativeNames = spark.table("Party_AlternativeNames")

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_final_AlternativeNames, party_AlternativeName_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_final_AlternativeNames, party_AlternativeName_dataobject, radar_datamodel_version_number, environment)